# 0 — PyTorch for beginners

Welcome! Before we dive into the history and mathematics of neural networks (Lectures 1 → x), we need a practical toolbox. That toolbox is **PyTorch**.

This first notebook is entirely about **the tensor** — PyTorch's central data structure — and everything you can do with it. No machine learning, no "learning", no *magic* yet. Just a thorough tour of the tool.

By the end you will be able to:

1. Create tensors and reason about their **shape**, **dtype**, and **device**.
2. Use **indexing** — basic, boolean, and integer-array — to pick out the values you need.
3. **Reshape** with `view`, `reshape`, `transpose`, `permute`, `cat`, `stack`, `split`.
4. Compute with **elementwise arithmetic** and **broadcasting**.
5. Apply **reductions** (`sum`, `mean`, `max`, `argmax`, `topk`, …) globally or along an axis.
6. Do standard **linear algebra** with `@` and `torch.linalg`.
7. Generate **random tensors** reproducibly, and **save / load** tensors from disk.

> **Who this notebook is for.** You know basic calculus and linear algebra, and you have some Python experience (ideally a bit of NumPy). You have *never* used PyTorch.

## How to run

- **On Google Colab (recommended).** `File → Upload notebook`, then `Runtime → Run all`. Everything you need (`torch`, `numpy`, `matplotlib`) is pre-installed.
- **Locally.** See the project `README.md` for `uv` or `pip` instructions.

No GPU is required.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("NumPy version:  ", np.__version__)
print("CUDA available: ", torch.cuda.is_available())

## 1. Tensors — the fundamental data structure

A **tensor** is PyTorch's name for a (possibly multidimensional) array of numbers. The **rank** (number of dimensions) determines what it looks like:

| Rank | Math object                | Example                                          |
| ---- | -------------------------- | ------------------------------------------------ |
| 0    | scalar $s \in \mathbb{R}$  | `torch.tensor(3.14)` — shape `()`                |
| 1    | vector $\vec{v}$           | `torch.tensor([1., 2., 3.])` — shape `(3,)`      |
| 2    | matrix $M$                 | `torch.zeros(2, 3)` — shape `(2, 3)`             |
| n    | n-d array                  | `torch.randn(4, 3, 2)` — shape `(4, 3, 2)`       |

For now, think of a tensor as **a NumPy array that also knows how to live on a GPU**. Almost every NumPy idiom carries over.


### 1.1 Creating tensors

The most common constructors:

- `torch.tensor(data)` — build from a Python list or NumPy array.
- `torch.zeros(shape)` / `torch.ones(shape)` — constant tensors.
- `torch.full(shape, value)` — constant tensor of any value.
- `torch.empty(shape)` — uninitialised memory (fast, but contents are garbage!).
- `torch.eye(n)` — identity matrix.
- `torch.arange(start, end, step)` — like `range`, but returns a tensor.
- `torch.linspace(start, end, steps)` — evenly-spaced points.
- `torch.randn(shape)` — random standard normal $\mathcal{N}(0, 1)$.
- `torch.rand(shape)` — random uniform on $[0, 1)$.
- `torch.randint(low, high, shape)` — random integers.

Two handy patterns: `torch.zeros_like(x)`, `torch.ones_like(x)`, `torch.full_like(x, value)` — same shape / dtype / device as `x`.


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.ones(3, 3)
d = torch.arange(0, 10, 2)
e = torch.linspace(-1.0, 1.0, 5)
f = torch.full((2, 2), 7.0)
g = torch.eye(3)
h = torch.randn(2, 3)
i = torch.randint(0, 10, (3, 3))

print("a =", a)
print("b =\n", b)
print("c =\n", c)
print("d =", d)
print("e =", e)
print("f =\n", f)
print("g =\n", g)
print("h =\n", h)
print("i =\n", i)

# _like variant
j = torch.zeros_like(h)
print("zeros_like(h) has shape", tuple(j.shape), "and dtype", j.dtype)

### 1.2 Tensor attributes

Every tensor carries metadata:

- `.shape` (or `.size()`) — the tuple of dimension sizes.
- `.dtype` — the element type.
- `.device` — where the tensor lives (`cpu` or `cuda:0`).
- `.ndim` (or `.dim()`) — the number of dimensions (rank).
- `.numel()` — total number of elements.


In [ ]:
x = torch.randn(3, 4)

print("shape  :", x.shape)
print("size() :", x.size())
print("dtype  :", x.dtype)
print("device :", x.device)
print("ndim   :", x.ndim)
print("numel  :", x.numel())

### 1.3 Dtypes — element types

Every tensor has a uniform **dtype**. The ones you'll meet most often:

| Dtype                     | Size    | Use                                                      |
| ------------------------- | ------- | -------------------------------------------------------- |
| `torch.float32` (default) | 4 bytes | Default for most computations                            |
| `torch.float64`           | 8 bytes | High precision (NumPy's default)                         |
| `torch.float16`           | 2 bytes | Half precision — smaller and faster on modern GPUs       |
| `torch.bfloat16`          | 2 bytes | Deep-learning-friendly 16-bit float                      |
| `torch.int64` (`long`)    | 8 bytes | Indices, class labels                                    |
| `torch.int32`             | 4 bytes | Integers                                                 |
| `torch.bool`              | 1 byte  | Boolean masks                                            |

Two things to remember:

1. **Operations usually require matching dtypes.** Mixing `int64` and `float32` will raise an error in most places.
2. **Convert with `.to(dtype)`** or with aliases like `.float()`, `.long()`, `.double()`, `.bool()`.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])  # defaults to float32
y = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
z = torch.tensor([1, 2, 3])  # defaults to int64

print("x.dtype =", x.dtype)
print("y.dtype =", y.dtype)
print("z.dtype =", z.dtype)

# Convert
print("x.double().dtype       =", x.double().dtype)
print("z.float().dtype        =", z.float().dtype)
print("x.to(torch.int32).dtype =", x.to(torch.int32).dtype)

# Booleans
mask = torch.tensor([True, False, True])
print("mask.dtype =", mask.dtype)
print("mask.sum() =", mask.sum().item())  # booleans sum as 0/1 -> count

### 1.4 The NumPy bridge

If you already know NumPy, you know most of PyTorch.

- `torch.from_numpy(array)` — NumPy → Tensor.
- `tensor.numpy()` — Tensor → NumPy (CPU tensors only).

The conversion is **zero-copy** when possible: both sides share the same memory, so modifying one modifies the other.


In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(arr)
print("as tensor:\n", t)

# They share memory!
t[0, 0] = 99.0
print("numpy array after tensor mutation:\n", arr)

### Exercise 1

Fill in the cell below to:

1. Create a tensor `x` of shape `(4, 5)` filled with standard-normal values, dtype `torch.float32`.
2. Create a tensor `y` of the same shape as `x` but filled with `3.14` (use `torch.full_like`).
3. Create a tensor `z` of 20 equally-spaced points between `-10` and `10`.
4. Print the shape and dtype of each.


In [ ]:
# 1. x of shape (4, 5), standard normal, float32
x = ...  # TODO

# 2. y full_like x, value 3.14
y = ...  # TODO

# 3. z = 20 equally-spaced points in [-10, 10]
z = ...  # TODO

# print shape and dtype of each
# ...

## 2. Indexing and selection

PyTorch supports three flavours of indexing — **basic**, **boolean**, and **integer-array** (a.k.a. "fancy") — that together let you pick out any subset of values from a tensor. Each is worth knowing.


### 2.1 Basic indexing and slicing

Standard Python / NumPy rules: integers pick a single element along an axis; slices `a:b:c` pick a range; `:` means "the whole axis"; `...` means "all remaining axes".


In [ ]:
X = torch.arange(20).view(4, 5).float()
print("X =\n", X)

print("X[0]       =", X[0])  # first row
print("X[:, 2]    =", X[:, 2])  # third column
print("X[1, 3]    =", X[1, 3].item())  # single element
print("X[1:3, :]  =\n", X[1:3, :])  # slice of rows
print("X[:, ::2]  =\n", X[:, ::2])  # every other column
print("X[..., -1] =", X[..., -1])  # last value along the last axis
print("X[0, ..., -1] =", X[0, ..., -1])

### 2.2 Boolean masks

You can index a tensor with a **boolean tensor of the same shape**. The result is a 1-D tensor containing only the values where the mask is `True`. Combining this with comparison operators is the idiomatic way to filter.


In [ ]:
X = torch.tensor([[1.0, -2.0, 3.0], [-4.0, 5.0, -6.0]])

mask = X > 0
print("mask =\n", mask)
print("X[mask] =", X[mask])  # all positives, flattened

# Replacement via boolean assignment (on a copy)
X_clipped = X.clone()
X_clipped[X_clipped < 0] = 0.0
print("X with negatives clipped to 0:\n", X_clipped)

### 2.3 Integer-array indexing

You can index with **another tensor of integer indices** to pick arbitrary rows, columns, or elements. This is how you implement "look up these specific items".


In [ ]:
X = torch.arange(20).view(4, 5).float()
print("X =\n", X)

# pick rows 0 and 2
print("rows [0, 2]:\n", X[torch.tensor([0, 2])])

# pick the diagonal: (0,0), (1,1), (2,2), (3,3)
idx = torch.arange(4)
print("diagonal:", X[idx, idx])

# repeat / reorder rows
print("rows [1, 1, 3, 0]:\n", X[torch.tensor([1, 1, 3, 0])])

### 2.4 Conditional selection: `torch.where` and `torch.clamp`

- `torch.where(condition, x, y)` — elementwise "if condition then x else y".
- `torch.clamp(x, min=..., max=...)` — clip values into a range.


In [ ]:
X = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])

# Keep positives, replace negatives with 0
print("where(X > 0, X, 0):", torch.where(X > 0, X, torch.zeros_like(X)))

# Clamp to [-1, 1]
print("clamp(X, -1, 1)   :", torch.clamp(X, min=-1.0, max=1.0))

### Exercise 2

Given `M = torch.randn(6, 6)`:

1. Extract the **second row** and the **last column** of `M`.
2. Extract only the **negative** entries of `M` (as a 1-D tensor).
3. In a **new** tensor, replace all entries of `M` whose absolute value is less than `0.5` with zero (don't mutate `M`).
4. Extract the **four corner values** of `M` in a single expression using integer-array indexing.


In [ ]:
torch.manual_seed(0)
M = torch.randn(6, 6)

# 1. second row, last column
# row   = ...
# col   = ...

# 2. negatives only
# negatives = ...

# 3. new tensor with small values zeroed
# M_cleaned = ...

# 4. four corners
# corners = ...

## 3. Reshaping and combining tensors

Changing a tensor's shape without changing its data is the most common operation in PyTorch code. Several tools exist, each with subtly different semantics.


### 3.1 `view` vs `reshape`

- `x.view(new_shape)` — returns a **view** that shares memory with `x`. Requires the memory to be *contiguous*. Fast and cheap.
- `x.reshape(new_shape)` — like `view` when possible, otherwise falls back to a copy. Safer default.

Use `view` when you know the tensor is contiguous (most freshly-created tensors are); use `reshape` when you're not sure. Pass `-1` in one position to let PyTorch **infer** that dimension.


In [ ]:
x = torch.arange(12)

A = x.view(3, 4)
print("A =\n", A)

# -1 means "infer this dimension"
B = x.view(-1, 6)
print("B shape:", B.shape)

# Flatten back to 1-D
print("flatten:", A.view(-1))

### 3.2 `transpose` and `permute`

- `x.transpose(i, j)` — swap two axes.
- `x.permute(a, b, c, ...)` — full axis reordering (the generalisation).
- `x.T` — shorthand for transposing a 2-D tensor.


In [ ]:
X = torch.arange(6).view(2, 3)
print("X =\n", X)
print("X.T =\n", X.T)
print("X.transpose(0, 1) =\n", X.transpose(0, 1))

# Higher-dim: (batch, channels, height, width) -> (batch, height, width, channels)
img = torch.randn(4, 3, 8, 8)
img_reordered = img.permute(0, 2, 3, 1)
print("img.shape          =", tuple(img.shape))
print("img_reordered.shape =", tuple(img_reordered.shape))

### 3.3 `squeeze` and `unsqueeze` — managing size-1 dimensions

- `x.unsqueeze(dim)` — insert a new size-1 dimension at position `dim`.
- `x.squeeze(dim)` — remove a size-1 dimension at `dim` (or all of them if `dim` is omitted).

These are the standard way to **align shapes for broadcasting** (Section 4).


In [ ]:
v = torch.tensor([1.0, 2.0, 3.0])
print("v               :", tuple(v.shape))
print("v.unsqueeze(0)  :", tuple(v.unsqueeze(0).shape))  # (1, 3) — row vector
print("v.unsqueeze(1)  :", tuple(v.unsqueeze(1).shape))  # (3, 1) — column vector

w = torch.zeros(1, 3, 1)
print("w               :", tuple(w.shape))
print("w.squeeze()     :", tuple(w.squeeze().shape))  # (3,)
print("w.squeeze(dim=0):", tuple(w.squeeze(dim=0).shape))  # (3, 1)

### 3.4 Combining tensors: `cat`, `stack`, `split`, `chunk`

- `torch.cat([a, b], dim=d)` — concatenate along an **existing** axis. All other axes must match.
- `torch.stack([a, b], dim=d)` — stack along a **new** axis. All tensors must have identical shape.
- `x.split(size, dim)` / `x.chunk(n, dim)` — the inverses: slice a tensor back into pieces.


In [ ]:
a = torch.zeros(2, 3)
b = torch.ones(2, 3)

# cat: along dim=0 gives (4, 3); along dim=1 gives (2, 6)
print("cat dim=0:\n", torch.cat([a, b], dim=0))
print("cat dim=1:\n", torch.cat([a, b], dim=1))

# stack: creates a new axis. Result is (2, 2, 3)
S = torch.stack([a, b], dim=0)
print("stack dim=0 shape:", tuple(S.shape))

# split into chunks of equal size along dim=1
x = torch.arange(12).view(3, 4)
parts = x.chunk(2, dim=1)
print("chunks:", [tuple(p.shape) for p in parts])

### Exercise 3

1. Create `v = torch.arange(6).float()`. Reshape it to shape `(2, 3)`, then transpose it. What is the resulting shape?
2. Create three random tensors of shape `(4, 4)`. Stack them into a single tensor of shape `(3, 4, 4)` using `torch.stack`. Separately, concatenate them side-by-side into a single tensor of shape `(4, 12)` using `torch.cat`.
3. Take a tensor `X = torch.arange(30).view(10, 3)` and split it along the first axis into **five** tensors of shape `(2, 3)`.


In [ ]:
# 1. arange -> view(2, 3) -> transpose
# v = ...
# reshaped = ...
# transposed = ...
# print("shape:", transposed.shape)

# 2. three random (4, 4); stack to (3, 4, 4); cat to (4, 12)
# A, B, C = ...
# stacked = ...
# concat  = ...

# 3. split X into five (2, 3) pieces
# X = ...
# parts = ...

## 4. Arithmetic and broadcasting

### 4.1 Elementwise operations

All standard arithmetic operators work elementwise on tensors of matching shape. PyTorch also ships the full suite of math functions as `torch.*`.


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print("a + b   =", a + b)
print("a * b   =", a * b)
print("a / b   =", a / b)
print("a ** 2  =", a**2)

# Math functions
print("torch.exp(a)  =", torch.exp(a))
print("torch.log(b)  =", torch.log(b))
print("torch.sqrt(b) =", torch.sqrt(b))
print("torch.sin(a)  =", torch.sin(a))

### 4.2 Broadcasting — the rules

When tensor shapes don't match, PyTorch tries to **broadcast** them into a compatible shape. The rules are identical to NumPy's:

1. Align shapes from the **right**.
2. Two dimensions are compatible if they are **equal** or if **one of them is 1**.
3. Any missing leading dimensions are treated as `1`.

Examples:

| Shapes                 | Result       | Why                       |
| ---------------------- | ------------ | ------------------------- |
| `(3,)` and `(3,)`      | `(3,)`       | identical                 |
| `(3,)` and `()`        | `(3,)`       | scalar broadcasts         |
| `(2, 3)` and `(3,)`    | `(2, 3)`     | `(3,)` becomes `(1, 3)`   |
| `(2, 3)` and `(2, 1)`  | `(2, 3)`     | the 1 expands to 3        |
| `(2, 3)` and `(2,)`    | **error**    | last dims `3` vs `2`      |

No data is ever actually copied — broadcasting is purely a shape-and-stride trick.


In [ ]:
M = torch.zeros(2, 3)
row = torch.tensor([1.0, 2.0, 3.0])  # shape (3,)
col = torch.tensor([[10.0], [20.0]])  # shape (2, 1)

print("M + row =\n", M + row)  # row added to every row
print("M + col =\n", M + col)  # col added to every column
print("M + row + col =\n", M + row + col)

# A classic use: centering a dataset column-by-column
data = torch.randn(100, 5)  # (N, features)
means = data.mean(dim=0, keepdim=True)  # (1, features)
centered = data - means  # broadcasts back to (100, 5)
print("mean after centering (should be ~0):", centered.mean(dim=0))

### 4.3 In-place operations (trailing underscore)

Most tensor operations have an **in-place** counterpart whose name ends with an underscore `_`. They modify the tensor rather than returning a new one, which can save memory:

- `x.add_(y)` is like `x = x + y` but reuses `x`'s memory.
- `x.zero_()`, `x.fill_(value)`, `x.clamp_(min, max)`, `x.abs_()`, …

Use in-place ops sparingly — they're easy to get wrong when multiple references point to the same tensor.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print("before: x =", x)
x.add_(y)  # x is now x + y
print("after x.add_(y):", x)

x.zero_()
print("after x.zero_():", x)

x.fill_(7.0)
print("after x.fill_(7.0):", x)

### Exercise 4

1. Without writing any Python loop, build a `10 × 10` **multiplication table** where `T[i, j] = (i+1) * (j+1)`. *Hint: broadcast two `torch.arange` tensors.*
2. Given `X = torch.randn(100, 3)`, in a new tensor center each **column** (subtract the per-column mean) and scale to unit variance (divide by the per-column std). No Python loops.


In [ ]:
# 1. 10x10 multiplication table with broadcasting
# rows = ...
# cols = ...
# table = ...

# 2. center-and-scale X
torch.manual_seed(0)
X = torch.randn(100, 3)
# X_norm = ...
# print("mean:", X_norm.mean(dim=0))
# print("std :", X_norm.std(dim=0))

## 5. Reductions

A **reduction** collapses one or more dimensions of a tensor into a summary statistic. You've already met `.sum()` and `.mean()`; PyTorch has a whole family.


### 5.1 Global reductions

Called with no arguments, these collapse the entire tensor to a scalar:

- `.sum()`, `.prod()`
- `.mean()`, `.std()`, `.var()`
- `.min()`, `.max()`
- `.all()`, `.any()` (for boolean tensors)


In [ ]:
torch.manual_seed(0)
X = torch.randn(3, 4)
print("X =\n", X)

print("sum  :", X.sum().item())
print("mean :", X.mean().item())
print("std  :", X.std().item())
print("min  :", X.min().item())
print("max  :", X.max().item())

# Boolean reductions
mask = X > 0
print("any positive?    ", mask.any().item())
print("all positive?    ", mask.all().item())
print("count positive:  ", mask.sum().item())

### 5.2 Reducing along a dimension: `dim` and `keepdim`

Pass `dim=d` to reduce along axis `d`. By default the reduced axis is **removed**; pass `keepdim=True` to keep it as a size-1 axis (handy for broadcasting back later).


In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("X.sum(dim=0)               :", X.sum(dim=0))  # (3,)
print("X.sum(dim=1)               :", X.sum(dim=1))  # (2,)
print("X.sum(dim=1, keepdim=True) :\n", X.sum(dim=1, keepdim=True))  # (2, 1)
print("X.mean(dim=0)              :", X.mean(dim=0))

### 5.3 `argmax`, `argmin`, `topk`, `sort`

- `x.argmax(dim)` / `x.argmin(dim)` — the **index** of the max / min along an axis.
- `x.topk(k, dim)` — the `k` largest values and their indices.
- `x.sort(dim)` — sorted values and their original indices.


In [ ]:
X = torch.tensor([[0.2, 0.9, 0.1, 0.6], [0.7, 0.3, 0.8, 0.4]])

print("argmax(dim=1) :", X.argmax(dim=1))
print("argmin(dim=1) :", X.argmin(dim=1))

values, indices = X.topk(2, dim=1)
print("top-2 values  :\n", values)
print("top-2 indices :\n", indices)

sorted_values, sorted_indices = X.sort(dim=1, descending=True)
print("sorted        :\n", sorted_values)

### Exercise 5

Let `X = torch.randn(50, 10)` (50 rows, 10 features).

1. Compute the **per-column** mean and std. What shape do they have?
2. Find the **row index** of the maximum value in each column.
3. For each row, find the indices of its **3 largest** values.


In [ ]:
torch.manual_seed(0)
X = torch.randn(50, 10)

# 1. per-column mean and std
# col_mean = ...
# col_std  = ...

# 2. row index of max per column
# row_of_max = ...

# 3. per-row top-3 indices
# top3_indices = ...

## 6. Linear algebra

### 6.1 Matrix and batched matrix products

- `A @ B` — standard matrix multiplication. Shape $(m, k) @ (k, n) \to (m, n)$.
- `torch.matmul` — same, but also supports **batched** inputs: the leading axes broadcast.
- `torch.einsum("...", x, y)` — fully general tensor contraction (advanced).


In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)

print("(A @ B).shape =", tuple((A @ B).shape))

# Batched matmul: 10 separate 3x4 @ 4x5 products -> 10 separate 3x5
batch_A = torch.randn(10, 3, 4)
batch_B = torch.randn(10, 4, 5)
print("batched matmul shape:", tuple(torch.matmul(batch_A, batch_B).shape))

### 6.2 A quick tour of `torch.linalg`

The `torch.linalg` submodule gives you the classical linear-algebra toolkit:

- `torch.linalg.norm(x)` — vector / matrix norms.
- `torch.linalg.inv(M)` — inverse.
- `torch.linalg.solve(A, b)` — solve $A x = b$ (preferred over `inv(A) @ b`: faster and numerically more stable).
- `torch.linalg.eig(M)` — eigenvalues and eigenvectors.
- `torch.linalg.svd(M)` — singular value decomposition.
- `torch.linalg.det(M)` — determinant.


In [ ]:
M = torch.tensor([[2.0, 1.0], [1.0, 3.0]])

print("norm(M)   =", torch.linalg.norm(M).item())
print("det(M)    =", torch.linalg.det(M).item())
print("inv(M)    =\n", torch.linalg.inv(M))

# Solve M x = b
b = torch.tensor([5.0, 10.0])
x = torch.linalg.solve(M, b)
print("solution to M x = b:", x)
print("check M @ x        :", M @ x)

### Exercise 6

1. Create a random matrix `A = torch.randn(4, 4)` and verify numerically that `A @ torch.linalg.inv(A)` is approximately the identity. *Tip: `torch.allclose(..., atol=1e-5)`.*
2. Compute the Euclidean norm of each **row** of `A`. *Hint: `torch.linalg.norm(A, dim=...)`.*
3. Solve the linear system `A x = b` where `b = torch.ones(4)`.


In [ ]:
torch.manual_seed(0)
A = torch.randn(4, 4)

# 1. A @ inv(A) ≈ I
# ...

# 2. row-wise Euclidean norms
# row_norms = ...

# 3. solve A x = 1
# x = ...

## 7. Random tensors and reproducibility

Random numbers are everywhere in numerics. Two things matter:

1. Knowing the main **constructors** (`rand`, `randn`, `randint`, `randperm`, `bernoulli`).
2. Making your experiments **reproducible** with `torch.manual_seed`.


In [ ]:
torch.manual_seed(42)
print("first run :", torch.randn(3))

torch.manual_seed(42)
print("re-seeded :", torch.randn(3))  # identical to the first run

# Without re-seeding, you get fresh draws
print("fresh     :", torch.randn(3))

### 7.2 Other random generators

- `torch.rand(shape)` — uniform on $[0, 1)$.
- `torch.randn(shape)` — standard normal $\mathcal{N}(0, 1)$.
- `torch.randint(low, high, shape)` — integers in $[\text{low}, \text{high})$.
- `torch.randperm(n)` — a random permutation of $[0, \dots, n - 1]$.
- `torch.bernoulli(p)` — 0/1 draws from per-element probabilities.


In [ ]:
torch.manual_seed(0)
print("rand       :", torch.rand(5))
print("randn      :", torch.randn(5))
print("randint    :", torch.randint(0, 10, (5,)))
print("randperm(6):", torch.randperm(6))

# Use randperm to shuffle the rows of a matrix
X = torch.arange(20).view(5, 4)
perm = torch.randperm(X.shape[0])
print("shuffled rows:\n", X[perm])

## 8. Saving and loading tensors

Tensors — and later, whole models — are persisted with `torch.save` / `torch.load`. They use Python's `pickle` under the hood, so you can save any combination of tensors and Python objects (dictionaries, lists, etc.).


In [ ]:
X = torch.randn(3, 3)
torch.save(X, "/tmp/tensor.pt")

X_loaded = torch.load("/tmp/tensor.pt", weights_only=True)
print("loaded:\n", X_loaded)
print("equal?", torch.equal(X, X_loaded))

# You can also save a dict of tensors
bundle = {"X": torch.randn(2, 2), "y": torch.tensor([0, 1])}
torch.save(bundle, "/tmp/bundle.pt")
loaded = torch.load("/tmp/bundle.pt", weights_only=True)
print("bundle keys:", list(loaded.keys()))

## 9. Tensors on the GPU (a one-minute detour)

When deep-learning code runs fast, it is running on a GPU. Moving a tensor to another device is a single method call:

```python
x_gpu = x.to("cuda")   # GPU (if available)
x_cpu = x_gpu.to("cpu")
```

A GPU tensor cannot be combined in the same operation with a CPU tensor, so in practice we pick a `device` once and push everything there. In Colab, `Runtime → Change runtime type → GPU` gives you a free one.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

x = torch.randn(3, 3).to(device)
print("x lives on:", x.device)

## 10. Wrap-up — what you can now do

You have a working command of PyTorch's central object:

- **Create** tensors (`tensor`, `zeros`, `arange`, `randn`, …) and inspect them (`shape`, `dtype`, `device`).
- **Index** them three ways: basic slices, boolean masks, integer-array indexing.
- **Reshape** with `view`, `reshape`, `transpose`, `permute`, `cat`, `stack`, `split`.
- **Compute** with elementwise ops and broadcasting.
- **Summarise** with reductions (`sum`, `mean`, `argmax`, `topk`, …).
- **Solve** linear-algebra problems with `@` and `torch.linalg`.
- **Reproduce** experiments with `torch.manual_seed` and persist results with `torch.save`.

### What's next

So far, a tensor is just a fast, GPU-aware NumPy array. In the next notebook we'll flip on the one extra feature that turns PyTorch from *a numerical library* into *the tool everyone uses for training neural networks* — and you'll start to see why the rest of the course is built around it.

For now, go and break things. Try every operation on your own tensors, change shapes, mix dtypes, see what errors you get. Getting comfortable here will pay for itself many times over from Lecture 1 onwards.
